# Stakeholder Analysis for LST Modeling Project

**Notebook 02: Stakeholder Distribution Mapping & Perception Analysis**

## What This Notebook Does

This notebook analyzes **stakeholder survey data** from workshop participants across Nigerian cities to understand the geographic distribution of project stakeholders, their professional backgrounds, and their perceptions of urban temperature changes. The results provide a human dimension to complement the satellite-derived LST analysis in subsequent notebooks.

---

## Table of Contents

Navigate directly to any section:

1. [Setup & Imports](#setup)
2. [Load Required Datasets](#load-data)
3. [Data Preparation & Aggregation](#data-prep)
4. [Create Geospatial Data](#geospatial)
5. [Prepare Data for Charts](#chart-data)
6. [Stakeholder Distribution Map](#stakeholder-map)
7. [City Classification Donut Chart](#donut-chart)
8. [Stakeholder Perception Analysis](#perception)
9. [Export Map and Data](#export)
10. [Summary & Next Steps](#summary)

---

## Sub-Research Objectives

1. **Map stakeholder geographic distribution** across 50 Nigerian cities with population context
2. **Characterize professional composition** of workshop attendees (academic, practitioner, GIS expert)
3. **Classify city engagement levels** (target vs. off-target, registered vs. unregistered, attended vs. not)
4. **Analyze stakeholder perceptions** on temperature increase and diurnal heat severity
5. **Generate publication-ready visualizations** combining spatial distribution with stakeholder metrics

---

## Data Sources

| Dataset | Source | Purpose |
|---------|--------|---------|
| **Nigeria Boundary** | GADM / Natural Earth (GeoJSON) | National boundary for map context |
| **Cities Population** | UN World Urbanization Prospects (CSV) | City locations and 2024 population estimates |
| **Stakeholder Survey** | Master Composite (CSV) | Registration, workshop attendance, perceptions |

---

## Notebook Structure

### Processing Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│ SECTION 1: Setup & Imports                                  │
│ • Load libraries and project configuration                  │
│ • Import modularized functions from src/                    │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 2: Load Required Datasets                           │
│ • Nigeria boundary GeoJSON                                  │
│ • Cities population CSV                                     │
│ • Stakeholder survey master composite                       │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 3: Data Preparation                                 │
│ • Count registered stakeholders per city                    │
│ • Filter workshop attendees (survey status = "Yes")         │
│ • Aggregate profession data                                 │
│ • Verify PID uniqueness and multi-city associations         │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 4: Geospatial & Chart Preparation                   │
│ • Convert cities to GeoDataFrame with population            │
│ • Merge stakeholder counts and attendance data              │
│ • Prepare profession dictionaries for pie charts            │
│ • Assign consistent color palette to professions            │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 5: Visualization                                    │
│ • Stakeholder distribution map with bar chart inset         │
│ • 3-ring donut chart (target / registered / attended)       │
│ • Perception analysis (warming + diurnal heat)              │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 6: Export                                           │
│ • Save maps as PNG + PDF                                    │
│ • Export stakeholder GeoDataFrame and profession data       │
│ • Export perception charts and summary tables               │
└─────────────────────────────────────────────────────────────┘
```

### Code Organization

- **`data_io.py`**: `load_stakeholder_datasets()`, `export_stakeholder_outputs()`, `export_perception_charts()`
- **`visualization.py`**: `create_stakeholder_map_with_bar_chart()`, `create_city_classification_donut()`, `create_perception_chart_with_donuts()`

## Output Structure

All results are saved to: `data/outputs/02_stakeholder_analysis/`

```
02_stakeholder_analysis/
├── Stakeholder_Distribution_Map_2024.png/pdf
├── City_Classification_Donut_2024.png/pdf
├── Perception_Analysis_2024.png/pdf
├── Perception_Donuts_2024.png/pdf
├── stakeholder_cities_2024.csv
└── workshop_professions_2024.csv
```

---

<a id="setup"></a>
## 1. Setup & Imports

Load project libraries, configuration, and visualization utilities.

**What This Section Does**:
- Imports core Python packages (`pandas`, `geopandas`, `matplotlib`, etc.)
- Loads modularized functions from `src/data_io.py` and `src/visualization.py`
- Initializes project configuration from `config.py`

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
"""
Stakeholder Analysis - Setup and Configuration
"""

# Core imports
import sys
from pathlib import Path

# Add project root and src to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# Import configuration
import config

# Import custom modules (consolidated in visualization.py)
from data_io import load_stakeholder_datasets, export_stakeholder_outputs, export_perception_charts
from visualization import (
    create_stakeholder_map_with_bar_chart,
    create_city_classification_donut,
    create_perception_chart_with_donuts
)

# Data processing libraries
import pandas as pd
import geopandas as gpd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import seaborn as sns

# Spatial operations
from shapely.geometry import Point

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("All modules imported successfully!")
print(f"Project root: {config.PROJECT_ROOT}")
print(f"Study year: {config.PROJECT_STUDY_YEAR}")
print(f"✅ Custom modules loaded: data_io, visualization")

<a id="load-data"></a>
## 2. Load Required Datasets

**Purpose**: Load the three key datasets for stakeholder analysis.

**What This Section Does**:
1. **Nigeria boundary** (GeoJSON) — for map background
2. **Major cities population data** (CSV) — city coordinates and 2024 population
3. **Stakeholder survey master composite** (CSV) — registration, attendance, perceptions

**Expected Output**: Three DataFrames ready for aggregation and mapping

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Load datasets using modularized function
boundary_gdf, cities_df, survey_df = load_stakeholder_datasets(config)

# Display sample of each dataset
print("\n" + "="*80)
print("CITIES DATA SAMPLE:")
print(cities_df[['Urban Agglomeration', 'Latitude', 'Longitude', str(config.PROJECT_STUDY_YEAR)]].head())

print("\n" + "="*80)
print("SURVEY DATA SAMPLE:")
print(survey_df[['pid', 'matched_city_x', 'matched_city_y', '45_disc_prof', 'survey status']].head())

<a id="data-prep"></a>
## 3. Data Preparation & Aggregation

**Purpose**: Clean and aggregate survey data for visualization.

**What This Section Does**:
- Counts unique registered stakeholders per city (`matched_city_x`)
- Filters workshop attendees (survey status = "Yes")
- Aggregates profession data per city
- Verifies PID uniqueness and checks for multi-city associations

**Expected Output**: Aggregated counts and profession breakdowns per city

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Clean and prepare survey data
print("Preparing stakeholder data...\n")

# 1. Count unique registered stakeholders per city (matched_city_x - cities they're associated with)
registered_counts = survey_df.groupby('matched_city_x')['pid'].nunique().reset_index()
registered_counts.columns = ['city', 'registered_count']
print(f"✅ Registered stakeholders aggregated: {len(registered_counts)} cities")
print(f"   Total unique stakeholders: {survey_df['pid'].nunique()}")

# 2. Filter workshop attendees (survey status = "Yes")
workshop_df = survey_df[survey_df['survey status'] == 'Yes'].copy()
print(f"\n✅ Workshop attendees filtered: {workshop_df['pid'].nunique()} unique participants")

# 3. Aggregate workshop attendees by city and profession
# Count unique pid per city and profession combination
workshop_prof = workshop_df.groupby(['matched_city_y', '45_disc_prof'])['pid'].nunique().reset_index()
workshop_prof.columns = ['city', 'profession', 'count']

# Get total attendees per city
workshop_total = workshop_df.groupby('matched_city_y')['pid'].nunique().reset_index()
workshop_total.columns = ['city', 'total_attendees']

print(f"\n✅ Workshop data aggregated:")
print(f"   Cities with attendees: {len(workshop_total)}")
print(f"   Profession categories: {workshop_prof['profession'].nunique()}")

# Display top cities by registered stakeholders
print("\n" + "="*80)
print("TOP 10 CITIES BY REGISTERED STAKEHOLDERS:")
print(registered_counts.sort_values('registered_count', ascending=False).head(10))

# Display top cities by workshop attendance
print("\n" + "="*80)
print("TOP 10 CITIES BY WORKSHOP ATTENDANCE:")
print(workshop_total.sort_values('total_attendees', ascending=False).head(10))

# Display profession distribution
print("\n" + "="*80)
print("PROFESSION DISTRIBUTION (Workshop Attendees):")
prof_summary = workshop_prof.groupby('profession')['count'].sum().sort_values(ascending=False)
print(prof_summary)

In [ ]:
# VERIFICATION: Check if PIDs appear in multiple cities
print("="*80)
print("DATA VERIFICATION")
print("="*80)

# Check for duplicate PIDs across cities
# Note: matched_city_x shows cities stakeholders are REGISTERED with/know
#       matched_city_y shows where workshops were ATTENDED
pid_city_counts_x = survey_df.groupby('pid')['matched_city_x'].nunique()
pids_multiple_cities_x = pid_city_counts_x[pid_city_counts_x > 1]

pid_city_counts_y = survey_df.groupby('pid')['matched_city_y'].nunique()
pids_multiple_cities_y = pid_city_counts_y[pid_city_counts_y > 1]

print(f"\n📊 Registration (matched_city_x - cities known/associated):")
if len(pids_multiple_cities_x) > 0:
    print(f"   ⚠️  {len(pids_multiple_cities_x)} PIDs associated with multiple cities")
    print(f"   (This is expected - stakeholders can know multiple cities)")
    print(f"   Examples:")
    for pid in pids_multiple_cities_x.head(3).index:
        cities = survey_df[survey_df['pid'] == pid]['matched_city_x'].unique()
        print(f"      PID {pid}: {', '.join(cities)}")
else:
    print(f"   ✅ All PIDs unique to single cities")

print(f"\n📊 Workshop Attendance (matched_city_y - where workshops attended):")
if len(pids_multiple_cities_y) > 0:
    print(f"   ⚠️  {len(pids_multiple_cities_y)} PIDs attended workshops in multiple cities")
    print(f"   Examples:")
    for pid in pids_multiple_cities_y.head(3).index:
        cities = survey_df[survey_df['pid'] == pid]['matched_city_y'].unique()
        print(f"      PID {pid}: {', '.join(cities)}")
else:
    print(f"   ✅ All PIDs attended workshop in single city")

# Verify total counts
print(f"\n✅ Total unique PIDs in survey_df: {survey_df['pid'].nunique()}")
print(f"✅ Sum of registered_counts per city: {registered_counts['registered_count'].sum()}")
print(f"✅ Total unique PIDs who attended workshop: {workshop_df['pid'].nunique()}")
print(f"✅ Sum of workshop_total per city: {workshop_total['total_attendees'].sum()}")

if registered_counts['registered_count'].sum() != survey_df['pid'].nunique():
    print(f"\n⚠️  NOTE: Sum of registered city counts ({registered_counts['registered_count'].sum()}) != Total unique PIDs ({survey_df['pid'].nunique()})")
    print("   This is expected when PIDs are associated with multiple cities (matched_city_x)")
else:
    print(f"\n✅ Registration counts match: Each PID appears in exactly one city")

print("="*80)

<a id="geospatial"></a>
## 4. Create Geospatial Data

**Purpose**: Convert city data into GeoDataFrames and merge with stakeholder statistics.

**What This Section Does**:
- Creates point geometries from city lat/lon coordinates
- Merges registered stakeholder counts per city
- Merges workshop attendance counts per city
- Creates a single GeoDataFrame with all spatial + stakeholder attributes

**Expected Output**: `cities_gdf` with population, registration, and attendance columns

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Create GeoDataFrame from cities with population data
print("Creating geospatial datasets...\n")

# Prepare cities GeoDataFrame
cities_geo = cities_df[['Urban Agglomeration', 'Latitude', 'Longitude', str(config.PROJECT_STUDY_YEAR)]].copy()
cities_geo.columns = ['city', 'latitude', 'longitude', 'population_2024']
cities_geo = cities_geo.dropna(subset=['latitude', 'longitude'])

# Create geometry
geometry = [Point(xy) for xy in zip(cities_geo['longitude'], cities_geo['latitude'])]
cities_gdf = gpd.GeoDataFrame(cities_geo, geometry=geometry, crs=config.WGS84)

print(f"✅ Cities GeoDataFrame created: {len(cities_gdf)} cities")

# Merge with registered stakeholders count
cities_gdf = cities_gdf.merge(registered_counts, on='city', how='left')
cities_gdf['registered_count'] = cities_gdf['registered_count'].fillna(0).astype(int)

# Merge with workshop attendees count
cities_gdf = cities_gdf.merge(workshop_total, on='city', how='left')
cities_gdf['total_attendees'] = cities_gdf['total_attendees'].fillna(0).astype(int)

print(f"\n✅ Merged with stakeholder data:")
print(f"   Cities with registered stakeholders: {(cities_gdf['registered_count'] > 0).sum()}")
print(f"   Cities with workshop attendees: {(cities_gdf['total_attendees'] > 0).sum()}")

# Display cities with stakeholder data
print("\n" + "="*80)
print("CITIES WITH STAKEHOLDER DATA:")
stakeholder_cities = cities_gdf[cities_gdf['registered_count'] > 0].sort_values('registered_count', ascending=False)
print(stakeholder_cities[['city', 'population_2024', 'registered_count', 'total_attendees']].to_string(index=False))

<a id="chart-data"></a>
## 5. Prepare Data for Charts

**Purpose**: Build nested dictionaries for visualization functions.

**What This Section Does**:
- Creates `{city: {profession: count}}` dictionary for pie chart segments
- Identifies all unique profession categories
- Assigns consistent color palette (distinct primary colours) to each profession

**Expected Output**: `profession_data` dictionary and `profession_colors` mapping

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Prepare profession data for pie charts
print("Preparing profession data for visualization...\n")

# Create dictionary: {city: {profession: count}}
profession_data = {}
for city in workshop_total['city']:
    city_profs = workshop_prof[workshop_prof['city'] == city]
    profession_data[city] = dict(zip(city_profs['profession'], city_profs['count']))

print(f"✅ Profession data prepared for {len(profession_data)} cities")

# Get all unique professions for consistent color mapping
all_professions = sorted(workshop_prof['profession'].unique())
print(f"\n✅ Unique profession categories: {len(all_professions)}")
print("\nProfession categories:")
for i, prof in enumerate(all_professions, 1):
    count = workshop_prof[workshop_prof['profession'] == prof]['count'].sum()
    print(f"   {i}. {prof}: {count} attendees")

# Define color palette for professions (distinct primary colors)
profession_colors = {
    'Academic': '#1f77b4',  # Blue
    'Practitioner (Public/private)': '#ff7f0e',  # Orange
    'GIS expert': '#2ca02c',  # Green
    'Academic|Practitioner (Public/private)': '#d62728',  # Red
    'GIS expert|Practitioner (Public/private)': '#9467bd',  # Purple
    'Academic|GIS expert': '#8c564b',  # Brown
    'Academic|GIS expert|Practitioner (Public/private)': '#e377c2',  # Pink
}

# Assign colors to any remaining professions not in predefined list
remaining_colors = ['#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a']
color_idx = 0
for prof in all_professions:
    if prof not in profession_colors and pd.notna(prof):
        profession_colors[prof] = remaining_colors[color_idx % len(remaining_colors)]
        color_idx += 1

print(f"\n✅ Color palette assigned to {len(profession_colors)} profession categories")

<a id="stakeholder-map"></a>
## 6. Stakeholder Distribution Map

**Purpose**: Create a comprehensive map showing stakeholder spatial distribution across Nigeria.

**What This Section Does**:
- Generates a Nigeria base map with city population bubbles
- Overlays stakeholder registration and attendance markers
- Includes bar chart inset showing per-city profession breakdown
- Displays global registration and attendance totals in legend

**Expected Output**: Publication-ready stakeholder distribution map (A4 upper half)

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# VISUALIZATION USES CONSOLIDATED FUNCTION from src/visualization.py
# Get global unique counts for legend (correct totals)
global_registered_count = survey_df['pid'].nunique()
global_attendees_count = workshop_df['pid'].nunique()

# Create stakeholder map using modularized visualization function
stakeholder_map = create_stakeholder_map_with_bar_chart(
    cities_gdf=cities_gdf,
    boundary_gdf=boundary_gdf,
    workshop_prof=workshop_prof,
    profession_colors=profession_colors,
    survey_df=survey_df,
    workshop_total=workshop_total,
    global_registered=global_registered_count,
    global_attendees=global_attendees_count,
    figsize=(11.7, 5.85)  # Upper half of A4
)

plt.show()
print("\n✅ Stakeholder distribution map created successfully!")
print("   (Generated using consolidated visualization.py module)")

<a id="donut-chart"></a>
## 7. City Classification Donut Chart

**Purpose**: Visualize the multi-level engagement pipeline from target cities to workshop attendance.

**What This Section Does**:

Generate a standalone 3-ring donut chart showing:
- **Ring 1**: Target vs Off-Target cities
- **Ring 2**: Registration status (registered vs. unregistered)
- **Ring 3**: Workshop attendance status (attended vs. not attended)

**Expected Output**: Donut chart showing engagement funnel

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Create standalone city classification donut chart
print("Creating city classification donut chart...\n")

donut_fig = create_city_classification_donut(
    cities_gdf=cities_gdf,
    survey_df=survey_df,
    workshop_total=workshop_total,
    figsize=(5.3, 5.3)
)

plt.show()

<a id="perception"></a>
## 8. Stakeholder Perception Analysis: City Warming & Diurnal Heating

**Purpose**: Analyze workshop participants' perceptions of urban temperature changes.

**What This Section Does**:

Analyze two survey questions per city and overall:
1. **Temperature increase over 10 years** — Has the city gotten warmer?
2. **Diurnal severity of heat** — Is heat more severe during day or night?

Creates comprehensive visualizations with:
- Per-city stacked bar charts showing response distributions
- Overall donut charts summarizing aggregate perceptions
- Combined figure for publication

**Expected Output**: Perception analysis figure with bar charts and donuts

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
"""
Stakeholder Perception Analysis: Temperature Warming & Diurnal Heating Patterns
This visualization combines both survey questions into a comprehensive chart showing:
- Per-city breakdown (stacked bars)
- Overall summary (pie/donut charts)
"""

print("Preparing stakeholder perception data...\n")

# Get unique responses per participant (first response per pid)
unique_resp = workshop_df.groupby('pid').agg({
    'matched_city_y': 'first',
    '61_temp_increase_10yrs': 'first',
    '60_diurnal_severity_heat': 'first'
}).reset_index()

# Clean NaN values
unique_resp['61_temp_increase_10yrs'] = unique_resp['61_temp_increase_10yrs'].fillna('No response')
unique_resp['60_diurnal_severity_heat'] = unique_resp['60_diurnal_severity_heat'].fillna('No response')

print(f"✅ Unique workshop participants analyzed: {len(unique_resp)}")
print(f"   Cities represented: {unique_resp['matched_city_y'].nunique()}")

# Cross-tabulations for visualization
warming_by_city = pd.crosstab(
    unique_resp['matched_city_y'], 
    unique_resp['61_temp_increase_10yrs']
)
diurnal_by_city = pd.crosstab(
    unique_resp['matched_city_y'], 
    unique_resp['60_diurnal_severity_heat']
)

# Overall totals
warming_total = unique_resp['61_temp_increase_10yrs'].value_counts()
diurnal_total = unique_resp['60_diurnal_severity_heat'].value_counts()

print("\n" + "="*60)
print("TEMPERATURE INCREASE PERCEPTION (Has city warmed in 10 yrs?):")
print(warming_total)
print("\n" + "="*60)
print("DIURNAL HEATING SEVERITY (When is heat worst?):")
print(diurnal_total)

In [ ]:
# Create perception charts using modularized visualization function
# This creates:
#   1. Main chart with side-by-side bars and embedded donuts
#   2. Separate standalone donut charts for other use cases

perception_fig, perception_donuts_fig = create_perception_chart_with_donuts(
    unique_resp=unique_resp,
    warming_total=warming_total,
    diurnal_total=diurnal_total,
    min_participants=2,
    figsize=(14, 5.5),
    legend_scale=1.2
)

plt.show()
print("\n(Generated using modularized function from src/visualization.py)")

<a id="export"></a>
## 9. Export Map and Data

**Purpose**: Save all visualizations and aggregated data for downstream use.

**What This Section Does**:
- Exports stakeholder distribution map as PNG + PDF
- Exports donut chart and perception analysis figures
- Saves stakeholder GeoDataFrame and profession data as CSV
- Uses modularized export functions from `src/data_io.py`

**Expected Output**: Complete file set in `data/outputs/02_stakeholder_analysis/`

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# EXPORT USES MODULARIZED FUNCTIONS from src/data_io.py
# Create output directory
output_dir = config.get_output_dir('02_stakeholder_analysis')

# Export stakeholder map and related data
export_paths = export_stakeholder_outputs(
    stakeholder_map=stakeholder_map,
    cities_gdf=cities_gdf,
    workshop_prof=workshop_prof,
    output_dir=output_dir,
    study_year=config.PROJECT_STUDY_YEAR,
    figure_dpi=config.FIGURE_DPI,
    donut_fig=donut_fig
)

# Export perception charts using modularized function
perception_paths = export_perception_charts(
    perception_fig=perception_fig,
    donuts_fig=perception_donuts_fig,
    unique_resp=unique_resp,
    output_dir=output_dir,
    study_year=config.PROJECT_STUDY_YEAR,
    figure_dpi=config.FIGURE_DPI
)

<a id="summary"></a>
## Summary & Next Steps

[↑ Back to Table of Contents](#Table-of-Contents)

---

### Key Accomplishments

This notebook completed **stakeholder analysis and perception mapping** for the LST modelling project:

#### ✅ Data Processing
- Loaded and cleaned stakeholder survey data (master composite)
- Aggregated registration counts, workshop attendance, and profession breakdowns per city
- Verified PID uniqueness and multi-city associations

#### ✅ Geospatial Mapping
- Created comprehensive stakeholder distribution map with population bubbles
- Overlaid profession bar chart inset for cities with attendees
- Generated 3-ring donut chart showing engagement pipeline

#### ✅ Perception Analysis
- Analyzed temperature increase perceptions across workshop cities
- Examined diurnal heat severity perceptions (day vs. night)
- Created combined visualization with per-city and overall summaries

#### ✅ Data Export
- Saved all figures as PNG (300 DPI) and PDF (vector)
- Exported stakeholder GeoDataFrame and profession data as CSV

---

### Key Findings

- Stakeholder coverage spans multiple Nigerian cities with varying engagement levels
- Professional composition includes academics, practitioners, and GIS experts
- Perception data provides a valuable human complement to satellite-derived LST trends
- Agreement between perceptions and measured trends will be validated in Notebook 03

---

### Next Steps: Monthly & Annual LST Analysis (Notebook 03)

The stakeholder perception data feeds directly into **trend validation**:

#### Notebook 03: Monthly & Annual LST Temporal Analysis

**What It Does**:
- Extracts MODIS Terra/Aqua LST data (2006–2025)
- Computes temporal statistics and trend analysis
- Creates cross-city heatmaps and comparison visualizations
- **Validates measured LST trends against stakeholder perceptions** (Block 9)

**Why This Matters**:
- Stakeholder perceptions of warming are compared against actual temperature data
- Agreement/disagreement analysis informs communication strategies
- Combined evidence strengthens policy recommendations

---